# YOLOv8 Pet Detection
## Training on Kaggle Dog and Cat Detection Dataset

## Step 1: Install Dependencies

In [2]:
# !pip install -q torch torchvision torchaudio
# !pip install -q ultralytics opencv-python pyyaml matplotlib numpy

## Step 2: Imports

In [3]:
import os
import random
from pathlib import Path
from typing import Dict, List, Optional
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import yaml

try:
    import torch
    TORCH_AVAILABLE = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'torch: {torch.__version__}, device: {device}')
except ImportError:
    TORCH_AVAILABLE = False
    device = 'cpu'

plt.rcParams['figure.figsize'] = (14, 8)

torch: 2.6.0+cu124, device: cuda


## Step 3: Dataset Helper

In [14]:
class DatasetHelper:
    @staticmethod
    def copy_dataset(source: str, dest: str):
        src = Path(source)
        dst = Path(dest)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'copy: {source} -> {dest}')
    
    @staticmethod
    def split_dataset(dataset_dir: str, train_ratio: float = 0.7, val_ratio: float = 0.15):
        print('\nsplitting data into train/val/test...')
        p = Path(dataset_dir)
        images_dir = p / 'images'
        labels_dir = p / 'labels'
        
        all_imgs = []
        for ext in ['*.jpg', '*.png', '*.jpeg']:
            all_imgs.extend(images_dir.glob(f'**/{ext}'))
        
        all_imgs = list(set(all_imgs))
        if not all_imgs:
            print('error: no images found')
            return False
        
        print(f'found {len(all_imgs)} images')
        
        for split in ['train', 'val', 'test']:
            (images_dir / split).mkdir(parents=True, exist_ok=True)
            (labels_dir / split).mkdir(parents=True, exist_ok=True)
        
        random.seed(42)
        random.shuffle(all_imgs)
        
        n_train = int(len(all_imgs) * train_ratio)
        n_val = int(len(all_imgs) * val_ratio)
        
        splits = {
            'train': all_imgs[:n_train],
            'val': all_imgs[n_train:n_train + n_val],
            'test': all_imgs[n_train + n_val:]
        }
        
        for split_name, imgs in splits.items():
            for img in imgs:
                dst_img = images_dir / split_name / img.name
                if not dst_img.exists():
                    shutil.copy2(img, dst_img)
                txt_file = img.with_suffix('.txt')
                if txt_file.exists():
                    dst_txt = labels_dir / split_name / txt_file.name
                    if not dst_txt.exists():
                        shutil.copy2(txt_file, dst_txt)
        return True
    
    @staticmethod
    def check_splits(dataset_dir: str) -> Optional[Dict[str, int]]:
        stats = {}
        p = Path(dataset_dir) / 'images'
        for split in ['train', 'val', 'test']:
            split_dir = p / split
            if split_dir.exists():
                imgs = list(split_dir.glob('*.jpg')) + list(split_dir.glob('*.png'))
                stats[split] = len(imgs)
            else:
                stats[split] = 0
        total = sum(stats.values())
        print('\ndataset structure:')
        for split, count in stats.items():
            pct = (count / total * 100) if total > 0 else 0
            print(f'  {split:5s}: {count:4d} ({pct:5.1f}%)')
        print(f'  total:  {total} images')
        if stats['train'] == 0:
            print('\nerror: train set is empty!')
            return None
        if stats['val'] == 0:
            print('\nwarning: val set is empty')
        return stats


## Step 4: Dataset Manager

In [15]:
class PetDatasetManager:
    def __init__(self, dataset_root: str = 'pet_dataset'):
        self.root = Path(dataset_root)
        self.classes = ['cat', 'dog']
    
    def create_yaml(self):
        config = {
            'path': str(self.root.absolute()),
            'train': str(self.root / 'images' / 'train'),
            'val': str(self.root / 'images' / 'val'),
            'test': str(self.root / 'images' / 'test'),
            'nc': 2,
            'names': self.classes
        }
        yaml_path = self.root / 'data.yaml'
        with open(yaml_path, 'w') as f:
            yaml.dump(config, f, sort_keys=False)
        print(f'created: {yaml_path}')
        return yaml_path

## Step 5: YOLO Trainer

In [16]:
class YOLOTrainer:
    def __init__(self, model_name: str = 'yolov8n'):
        from ultralytics import YOLO
        self.YOLO = YOLO
        self.model_name = model_name
        self.model = None
    
    def load_model(self):
        self.model = self.YOLO(f'{self.model_name}.pt')
        print(f'model {self.model_name} loaded')
    
    def train(self, yaml_path: str, epochs: int = 20, batch_size: int = 8, 
              imgsz: int = 320, device: int = -1, project: str = 'yolo_runs'):
        if not self.model:
            self.load_model()
        print(f'\ntraining params:')
        print(f'  epochs: {epochs}')
        print(f'  batch: {batch_size}')
        print(f'  size: {imgsz}x{imgsz}')
        print(f'  device: {"GPU" if device >= 0 else "CPU"}')
        print(f'\nstarting training...')
        results = self.model.train(
            data=yaml_path, epochs=epochs, imgsz=imgsz, batch=batch_size,
            device=device, project=project, name='pet_model', patience=10,
            save=True, exist_ok=True, verbose=True)
        print('\ntraining completed')
        return results
    
    def save(self, path: str = 'pet_model.pt'):
        if self.model:
            self.model.save(path)
            print(f'model saved: {path}')

## Step 6: Detector

In [17]:
class Detector:
    def __init__(self, model_path: str):
        from ultralytics import YOLO
        self.model = YOLO(model_path)
        self.classes = ['cat', 'dog']
        np.random.seed(42)
        self.colors = {i: tuple(np.random.randint(50, 255, 3).tolist()) 
                      for i in range(len(self.classes))}
        print(f'model loaded: {model_path}')
    
    def detect(self, image_path: str, conf: float = 0.5) -> List[Dict]:
        img = cv2.imread(image_path)
        if img is None:
            return []
        results = self.model.predict(img, conf=conf)
        detections = []
        for result in results:
            for box in result.boxes:
                detections.append({
                    'class': self.classes[int(box.cls[0])],
                    'conf': float(box.conf[0]),
                    'bbox': tuple(float(x) for x in box.xyxy[0])
                })
        return detections
    
    def visualize(self, image_path: str, detections: List[Dict], output_path: str = None):
        img = cv2.imread(image_path)
        if img is None:
            return
        for det in detections:
            x1, y1, x2, y2 = map(int, det['bbox'])
            color = self.colors[self.classes.index(det['class'])]
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            label = f"{det['class']} {det['conf']:.2f}"
            cv2.putText(img, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        if output_path:
            cv2.imwrite(output_path, img)
            print(f'saved: {output_path}')

## Step 7: Prepare Data

In [18]:
source_folder = 'annotations'
target_folder = 'pet_dataset_final'
print(f'\nchecking dataset in {source_folder}...')
if not Path(source_folder).exists():
    print(f'error: folder {source_folder} not found')
else:
    DatasetHelper.copy_dataset(source_folder, target_folder)
    DatasetHelper.split_dataset(target_folder)
    stats = DatasetHelper.check_splits(target_folder)
    if stats:
        print('\ndata ready for training')
    else:
        print('\nerror: dataset structure problem')


checking dataset in annotations...
copy: annotations -> pet_dataset_final

splitting data into train/val/test...
found 3686 images

dataset structure:
  train: 2580 ( 70.0%)
  val  :  552 ( 15.0%)
  test :  554 ( 15.0%)
  total:  3686 images

data ready for training


In [19]:
# manager = PetDatasetManager(target_folder)
# yaml_path = manager.create_yaml()

created: pet_dataset_final\data.yaml


In [43]:
from pathlib import Path
import xml.etree.ElementTree as ET
import cv2

ANNOT_DIR = Path(r"D:/hse_cv2/annotations/annotations")          
IMG_ROOT  = Path(r"D:/hse_cv2/pet_dataset_final/images")         
LBL_ROOT  = Path(r"D:/hse_cv2/pet_dataset_final/labels")         

for split in ["train", "val"]:
    (LBL_ROOT / split).mkdir(parents=True, exist_ok=True)

classes = {"cat": 0, "dog": 1}

def voc_to_yolo(xml_path: Path, img_w: int, img_h: int) -> str:
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in classes:
            continue
        cls_id = classes[cls_name]

        bbox = obj.find("bndbox")
        xmin = float(bbox.find("xmin").text)
        ymin = float(bbox.find("ymin").text)
        xmax = float(bbox.find("xmax").text)
        ymax = float(bbox.find("ymax").text)

        x_c = (xmin + xmax) / 2.0 / img_w
        y_c = (ymin + ymax) / 2.0 / img_h
        w   = (xmax - xmin) / img_w
        h   = (ymax - ymin) / img_h

        lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")

    return "\n".join(lines)

for split in ["train", "val"]:
    img_dir = IMG_ROOT / split
    lbl_dir = LBL_ROOT / split

    imgs = list(img_dir.glob("*.png")) + list(img_dir.glob("*.jpg"))
    print(f"{split} images: {len(imgs)}")

    created = 0
    for img_path in imgs:
        xml_path = ANNOT_DIR / f"{img_path.stem}.xml"
        if not xml_path.exists():
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        yolo_txt = voc_to_yolo(xml_path, w, h)
        if yolo_txt:
            (lbl_dir / f"{img_path.stem}.txt").write_text(yolo_txt, encoding="utf-8")
            created += 1

    print(f"{split} labels created: {created}")


train images: 2580
train labels created: 2580
val images: 552
val labels created: 552


## Step 8: Train Model

In [45]:
trainer = YOLOTrainer('yolov8n')
trainer.load_model()
device_id = 0 if torch.cuda.is_available() else -1
results = trainer.train(
    yaml_path=str(yaml_path),
    epochs=5,
    batch_size=4,
    imgsz=320,
    device=device_id,
    project='yolo_output'
)
trainer.save('pet_model_final.pt')

model yolov8n loaded

training params:
  epochs: 5
  batch: 4
  size: 320x320
  device: GPU

starting training...
Ultralytics 8.3.241  Python-3.10.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3070, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=pet_dataset_final\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=pet_model, nbs=64,

## Step 9: Test

In [46]:
test_dir = Path(target_folder) / 'images' / 'test'
if test_dir.exists() and len(list(test_dir.glob('*.*'))) > 0:
    test_images = list(test_dir.glob('*.jpg')) + list(test_dir.glob('*.png'))
    print(f'found {len(test_images)} test images\n')
    if test_images:
        detector = Detector('pet_model_final.pt')
        for img_path in test_images[:3]:
            detections = detector.detect(str(img_path), conf=0.5)
            if detections:
                print(f'{img_path.name}:')
                for det in detections:
                    print(f'  {det["class"]:6s} {det["conf"]:.2f}')
                detector.visualize(str(img_path), detections, f'test_result_{img_path.stem}.jpg')
                print()
            else:
                print(f'{img_path.name}: nothing found\n')
else:
    print('test set not found\n')

found 554 test images

model loaded: pet_model_final.pt

0: 320x320 2 dogs, 11.1ms
Speed: 14.8ms preprocess, 11.1ms inference, 1.8ms postprocess per image at shape (1, 3, 320, 320)
Cats_Test1.png:
  dog    0.93
  dog    0.90
saved: test_result_Cats_Test1.jpg


0: 256x320 1 dog, 18.5ms
Speed: 1.4ms preprocess, 18.5ms inference, 3.3ms postprocess per image at shape (1, 3, 256, 320)
Cats_Test10.png:
  dog    0.83
saved: test_result_Cats_Test10.jpg


0: 256x320 1 cat, 18.6ms
Speed: 1.2ms preprocess, 18.6ms inference, 1.6ms postprocess per image at shape (1, 3, 256, 320)
Cats_Test1002.png:
  cat    0.92
saved: test_result_Cats_Test1002.jpg

